# 13 — Kunnskapsgrafer og GraphRAG

**Fase:** 2 — Kjerne AI | **Tid:** 2 timer | **Krav:** Notatbok 08

**Hva du bygger:** En kunnskapsgraf over pensjonsbegreper, og et GraphRAG-system som kan svare på spørsmål som krever å navigere relasjoner — ikke bare finne lignende tekst.

---

## Når RAG ikke er nok

Semantisk søk er flott for å finne *lignende* innhold. Men noen spørsmål krever å følge *relasjoner*:

```
Q: "Hvilke pensjonstyper krever minst 3 år tjeneste?"
→ Vanlig RAG: Finner dokumenter om tjenestekrav, men kobler ikke alle typene
→ Kunnskapsgraf: AFP → [krever] → 3 år tjeneste  ✅

Q: "Hva er koblingen mellom AFP og alderspensjon?"
→ Vanlig RAG: Kan mangle eksplisitt kobling
→ Kunnskapsgraf: AFP → [kombineres_med] → Alderspensjon  ✅
```

**En graf består av:**
- **Noder** (entiteter): AFP, Alderspensjon, SPK, 62 år
- **Kanter** (relasjoner): `[gir_rett_til]`, `[krever]`, `[kombineres_med]`

In [ ]:
%pip install -q networkx matplotlib openai

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Bygg en rettet graf over pensjonsbegreper
G = nx.DiGraph()

# Noder
noder = [
    ("AFP",          {"type": "pensjonstype", "beskrivelse": "Avtalefestet pensjon"}),
    ("Alderspensjon",{"type": "pensjonstype", "beskrivelse": "Fra 67 år"}),
    ("Uførepensjon", {"type": "pensjonstype", "beskrivelse": "Ved nedsatt arbeidsevne"}),
    ("Barnepensjon", {"type": "pensjonstype", "beskrivelse": "Til barn under 20"}),
    ("SPK",          {"type": "organisasjon", "beskrivelse": "Statens pensjonskasse"}),
    ("NAV",          {"type": "organisasjon", "beskrivelse": "Arbeids- og velferdsetaten"}),
    ("62 år",        {"type": "alder"}),
    ("67 år",        {"type": "alder"}),
    ("3 år tjeneste",{"type": "krav"}),
    ("30 år tjeneste",{"type": "krav"}),
    ("20% nedsatt",  {"type": "krav"}),
]
G.add_nodes_from(noder)

# Kanter (relasjoner)
kanter = [
    ("SPK",         "AFP",          {"relasjon": "forvalter"}),
    ("SPK",         "Alderspensjon",{"relasjon": "forvalter"}),
    ("SPK",         "Uførepensjon", {"relasjon": "forvalter"}),
    ("NAV",         "Uførepensjon", {"relasjon": "behandler_søknad"}),
    ("AFP",         "62 år",        {"relasjon": "tidligst_fra"}),
    ("AFP",         "3 år tjeneste",{"relasjon": "krever"}),
    ("AFP",         "Alderspensjon",{"relasjon": "kombineres_med"}),
    ("Alderspensjon","67 år",       {"relasjon": "utbetales_fra"}),
    ("Alderspensjon","30 år tjeneste",{"relasjon": "full_sats_krever"}),
    ("Uførepensjon","20% nedsatt",  {"relasjon": "krever"}),
    ("Uførepensjon","Alderspensjon",{"relasjon": "konverteres_til_ved_67"}),
    ("Barnepensjon","20 år",        {"relasjon": "gjelder_til"}),
]
G.add_edges_from([(u, v, d) for u, v, d in kanter])

print(f"Graf: {G.number_of_nodes()} noder, {G.number_of_edges()} kanter")

In [ ]:
# Visualiser grafen
plt.figure(figsize=(14, 8))
pos = nx.spring_layout(G, seed=42, k=2)

farger = {
    "pensjonstype":  "#4e9af1",
    "organisasjon":  "#f1c94e",
    "alder":         "#4ef1a8",
    "krav":          "#f16e4e",
}
node_farger = [farger.get(G.nodes[n].get("type", ""), "#cccccc") for n in G.nodes]

nx.draw_networkx_nodes(G, pos, node_color=node_farger, node_size=1500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
nx.draw_networkx_edges(G, pos, edge_color="gray", arrows=True, arrowsize=15, width=1.5)
kant_etiketter = {(u, v): d["relasjon"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, kant_etiketter, font_size=7, label_pos=0.3)

plt.title("Pensjons-kunnskapsgraf", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

---

## Del 2: Spørre grafen

In [ ]:
def finn_relasjoner(node: str) -> list:
    """Finn alle kanter fra en node."""
    return [
        (node, d["relasjon"], nabo)
        for nabo, d in G[node].items()
    ]

def finn_hva_krever(krav: str) -> list:
    """Finn alle pensjonstyper som krever noe bestemt."""
    return [
        f"{kilde} krever {krav}"
        for kilde, mål, d in G.edges(data=True)
        if mål == krav and d["relasjon"] == "krever"
    ]

def korteste_sti(fra: str, til: str) -> list:
    """Finn koblingen mellom to begreper."""
    try:
        return nx.shortest_path(G, fra, til)
    except nx.NetworkXNoPath:
        return []

print("Relasjoner fra AFP:")
for rel in finn_relasjoner("AFP"):
    print(f"  {rel[0]} --[{rel[1]}]--> {rel[2]}")

print("\nHva krever 3 år tjeneste?")
for res in finn_hva_krever("3 år tjeneste"):
    print(f"  {res}")

print("\nKobling fra AFP til Alderspensjon:")
sti = korteste_sti("AFP", "Alderspensjon")
print(f"  {' → '.join(sti)}")

---

## Del 3: GraphRAG — Kombiner graf + LLM

In [ ]:
from openai import OpenAI

llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def graf_kontekst(spørsmål: str) -> str:
    """
    Enkel GraphRAG: finn relevante noder og hent deres relasjoner
    som strukturert kontekst til LLM.
    """
    # Finn noder som nevnes i spørsmålet
    relevante = [n for n in G.nodes if n.lower() in spørsmål.lower()]

    # Fallback: bruk pensjonstyper hvis ingenting matcher
    if not relevante:
        relevante = [n for n, d in G.nodes(data=True) if d.get("type") == "pensjonstype"]

    linjer = []
    for node in relevante:
        for nabo, data in G[node].items():
            linjer.append(f"{node} --[{data['relasjon']}]--> {nabo}")

    return "\n".join(linjer)

def graphrag_svar(spørsmål: str) -> str:
    kontekst = graf_kontekst(spørsmål)
    svar = llm.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system",
             "content": "Du er pensjonsrådgiver. Bruk grafrelasjonene nedenfor til å svare. Svar på norsk."},
            {"role": "user",
             "content": f"Grafkontekst:\n{kontekst}\n\nSpørsmål: {spørsmål}"},
        ],
        temperature=0.1,
    )
    return svar.choices[0].message.content

for q in [
    "Hva er koblingen mellom AFP og Alderspensjon?",
    "Hva kreves for å få Uførepensjon?",
]:
    print(f"Q: {q}")
    print(f"A: {graphrag_svar(q)}\n")

---

## Oppsummering

| Konsept | Verktøy | Passer for |
|---------|---------|----------|
| Graf | NetworkX (lokalt/gratis) | Prototyper |
| Grafspørring | `G[node]`, `shortest_path` | Relasjoner |
| Grafvisualsiering | Matplotlib | Forstå struktur |
| GraphRAG | Graf + LLM | Multi-hop spørsmål |
| Produksjon | Neo4j (gratis community) | Store grafer |

---

## Hva er neste steg?

**Fase 3 begynner! Neste: `14_data_pipelines_etl.ipynb`** — Nå som du kan bygge AI-systemer, lærer du å mate dem med data pålitelig. ETL-pipelines, datavalidering og automatisering.